# Modélisation — Prédiction de la consommation électrique (EDF / RTE)

**Pipeline** :
1. Feature engineering (cyclicité, lags, rolling, jours fériés)
2. Split temporel train / validation / test
3. Modèles : Baseline · Ridge · Random Forest · XGBoost
4. Tracking MLflow
5. Évaluation & comparaison

In [ ]:
# Diagnostic — vérifie que le bon Python est utilisé
import sys
print(sys.executable)
# Doit contenir : .../projet-edf/.venv/bin/python

## 0. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
from sklearn.pipeline import Pipeline

import xgboost as xgb
import mlflow

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (14, 5)

print('MLflow', mlflow.__version__)
print('Librairies chargées.')

## 1. Chargement des données

In [ ]:
df = pd.read_csv('data/daily_consumption.csv', parse_dates=['Date'])
df = df.sort_values('Date').reset_index(drop=True)

print(f'Période : {df["Date"].min().date()} → {df["Date"].max().date()}')
print(f'Shape   : {df.shape}')
df[['Date', 'conso_mean_mw']].tail()

## 2. Feature Engineering

### 2.1 Jours fériés français

In [ ]:
# Jours fériés français (sans dépendance externe)
def french_holidays(years):
    """Retourne un set de dates correspondant aux jours fériés fixes + Pâques."""
    from datetime import date, timedelta

    def easter(year):
        """Algorithme de Butcher pour la date de Pâques."""
        a = year % 19
        b, c = divmod(year, 100)
        d, e = divmod(b, 4)
        f = (b + 8) // 25
        g = (b - f + 1) // 3
        h = (19 * a + b - d - g + 15) % 30
        i, k = divmod(c, 4)
        l = (32 + 2 * e + 2 * i - h - k) % 7
        m = (a + 11 * h + 22 * l) // 451
        month, day = divmod(114 + h + l - 7 * m, 31)
        return date(year, month, day + 1)

    holidays = set()
    for y in years:
        e = easter(y)
        holidays |= {
            date(y, 1, 1),   # Jour de l'an
            e + timedelta(1), # Lundi de Pâques
            date(y, 5, 1),   # Fête du Travail
            date(y, 5, 8),   # Victoire 1945
            e + timedelta(39), # Ascension
            e + timedelta(50), # Lundi de Pentecôte
            date(y, 7, 14),  # Fête Nationale
            date(y, 8, 15),  # Assomption
            date(y, 11, 1),  # Toussaint
            date(y, 11, 11), # Armistice
            date(y, 12, 25), # Noël
        }
    return holidays

years = df['Date'].dt.year.unique()
holidays = french_holidays(years)
df['is_holiday'] = df['Date'].dt.date.apply(lambda d: int(d in holidays))

print(f"Jours fériés couverts (ex. 2023) : {sorted(d for d in holidays if d.year == 2023)}")

### 2.2 Encodage cyclique

In [ ]:
df['month_sin']   = np.sin(2 * np.pi * df['month']     / 12)
df['month_cos']   = np.cos(2 * np.pi * df['month']     / 12)
df['doy_sin']     = np.sin(2 * np.pi * df['dayofyear'] / 365)
df['doy_cos']     = np.cos(2 * np.pi * df['dayofyear'] / 365)
df['dow_sin']     = np.sin(2 * np.pi * df['dayofweek'] / 7)
df['dow_cos']     = np.cos(2 * np.pi * df['dayofweek'] / 7)

print('Encodage cyclique OK.')

### 2.3 Lag features & rolling means

In [ ]:
target = 'conso_mean_mw'

# Lags
for lag in [1, 7, 14, 30, 365]:
    df[f'lag_{lag}'] = df[target].shift(lag)

# Rolling means (sur la conso shiftée de 1 jour pour éviter la fuite)
df['roll_7']  = df[target].shift(1).rolling(7,  min_periods=1).mean()
df['roll_30'] = df[target].shift(1).rolling(30, min_periods=1).mean()

# Prévision J-1 RTE comme feature (baseline externe)
if 'prevision_j1_mean' in df.columns:
    df['prevision_j1_mean'] = df['prevision_j1_mean'].fillna(df['lag_1'])

# Supprimer les lignes avec NaN (dues aux lags)
df_feat = df.dropna().copy()
print(f'Dataset après feature engineering : {df_feat.shape[0]} jours')
df_feat[['Date', target, 'lag_1', 'lag_7', 'lag_365', 'roll_7', 'roll_30']].head()

## 3. Split temporel

| Période | Usage |
|---------|-------|
| 2013–2021 | Train |
| 2022 | Validation |
| 2023–2024 | Test |

In [ ]:
FEATURES = [
    # Temporel cyclique
    'month_sin', 'month_cos', 'doy_sin', 'doy_cos', 'dow_sin', 'dow_cos',
    # Catégoriel
    'is_weekend', 'is_holiday',
    # Lags
    'lag_1', 'lag_7', 'lag_14', 'lag_30', 'lag_365',
    # Rolling
    'roll_7', 'roll_30',
]

TARGET = 'conso_mean_mw'

train = df_feat[df_feat['Date'].dt.year <= 2021]
val   = df_feat[df_feat['Date'].dt.year == 2022]
test  = df_feat[df_feat['Date'].dt.year >= 2023]

X_train, y_train = train[FEATURES], train[TARGET]
X_val,   y_val   = val[FEATURES],   val[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

print(f'Train : {len(train)} jours ({train["Date"].min().date()} → {train["Date"].max().date()})')
print(f'Val   : {len(val)} jours ({val["Date"].min().date()} → {val["Date"].max().date()})')
print(f'Test  : {len(test)} jours ({test["Date"].min().date()} → {test["Date"].max().date()})')

## 4. Fonctions utilitaires

In [ ]:
def evaluate(y_true, y_pred, label=''):
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    r2   = r2_score(y_true, y_pred)
    if label:
        print(f'[{label}]  RMSE={rmse:.1f} MW  |  MAPE={mape:.2f}%  |  R²={r2:.4f}')
    return {'rmse': rmse, 'mape': mape, 'r2': r2}


def plot_predictions(dates, y_true, y_pred, title):
    fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=False)

    # Série temporelle
    axes[0].plot(dates, y_true.values,  label='Réel',       color='steelblue', lw=1.5)
    axes[0].plot(dates, y_pred,         label='Prédit',     color='firebrick', lw=1.2, alpha=0.8)
    axes[0].set_title(title, fontweight='bold')
    axes[0].set_ylabel('MW moyen journalier')
    axes[0].legend()
    axes[0].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha='right')

    # Résidus
    residuals = y_true.values - y_pred
    axes[1].bar(dates, residuals, color=np.where(residuals > 0, 'steelblue', 'coral'),
                width=1, alpha=0.7)
    axes[1].axhline(0, color='black', lw=0.8)
    axes[1].set_ylabel('Résidu (MW)')
    axes[1].set_title('Résidus (réel − prédit)', fontweight='bold')
    axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30, ha='right')

    plt.tight_layout()
    plt.show()

results = {}  # stocke les métriques test de chaque modèle
print('Utilitaires définis.')

## 5. Modèles

### 5.1 Baseline — Persistance lag-1 et lag-7

In [ ]:
# Baseline lag-1 : on prédit J avec la conso de J-1
metrics_lag1 = evaluate(y_test, test['lag_1'], 'Baseline lag-1 (test)')
results['Baseline lag-1'] = metrics_lag1

# Baseline lag-7 : même jour la semaine dernière
metrics_lag7 = evaluate(y_test, test['lag_7'], 'Baseline lag-7 (test)')
results['Baseline lag-7'] = metrics_lag7

# Prévision RTE J-1 si disponible
if 'prevision_j1_mean' in test.columns:
    metrics_rte = evaluate(y_test, test['prevision_j1_mean'], 'Benchmark RTE J-1 (test)')
    results['Benchmark RTE J-1'] = metrics_rte

### 5.2 Ridge Regression

In [ ]:
with mlflow.start_run(run_name='Ridge'):
    pipe_ridge = Pipeline([
        ('scaler', StandardScaler()),
        ('model',  Ridge(alpha=10.0))
    ])
    pipe_ridge.fit(X_train, y_train)

    pred_val_ridge  = pipe_ridge.predict(X_val)
    pred_test_ridge = pipe_ridge.predict(X_test)

    m_val  = evaluate(y_val,  pred_val_ridge,  'Ridge val')
    m_test = evaluate(y_test, pred_test_ridge, 'Ridge test')

    mlflow.log_params({'alpha': 10.0, 'model': 'Ridge'})
    mlflow.log_metrics({f'val_{k}': v for k, v in m_val.items()})
    mlflow.log_metrics({f'test_{k}': v for k, v in m_test.items()})
    mlflow.sklearn.log_model(pipe_ridge, 'ridge_model')

results['Ridge'] = m_test

plot_predictions(test['Date'], y_test, pred_test_ridge, 'Ridge — Test 2023–2024')

### 5.3 Random Forest

In [ ]:
RF_PARAMS = {
    'n_estimators': 300,
    'max_depth': 12,
    'min_samples_leaf': 5,
    'n_jobs': -1,
    'random_state': 42,
}

with mlflow.start_run(run_name='RandomForest'):
    rf = RandomForestRegressor(**RF_PARAMS)
    rf.fit(X_train, y_train)

    pred_val_rf  = rf.predict(X_val)
    pred_test_rf = rf.predict(X_test)

    m_val  = evaluate(y_val,  pred_val_rf,  'RF val')
    m_test = evaluate(y_test, pred_test_rf, 'RF test')

    mlflow.log_params(RF_PARAMS)
    mlflow.log_metrics({f'val_{k}': v for k, v in m_val.items()})
    mlflow.log_metrics({f'test_{k}': v for k, v in m_test.items()})
    mlflow.sklearn.log_model(rf, 'rf_model')

results['RandomForest'] = m_test

plot_predictions(test['Date'], y_test, pred_test_rf, 'Random Forest — Test 2023–2024')

### 5.4 XGBoost

In [ ]:
XGB_PARAMS = {
    'n_estimators':      500,
    'learning_rate':     0.05,
    'max_depth':         6,
    'subsample':         0.8,
    'colsample_bytree':  0.8,
    'reg_alpha':         0.1,
    'reg_lambda':        1.0,
    'random_state':      42,
    'n_jobs':            -1,
    'early_stopping_rounds': 30,
    'eval_metric':       'rmse',
}

with mlflow.start_run(run_name='XGBoost'):
    xgb_model = xgb.XGBRegressor(**XGB_PARAMS)
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=50
    )

    pred_val_xgb  = xgb_model.predict(X_val)
    pred_test_xgb = xgb_model.predict(X_test)

    m_val  = evaluate(y_val,  pred_val_xgb,  'XGBoost val')
    m_test = evaluate(y_test, pred_test_xgb, 'XGBoost test')

    mlflow.log_params({k: v for k, v in XGB_PARAMS.items() if k != 'early_stopping_rounds'})
    mlflow.log_param('early_stopping_rounds', XGB_PARAMS['early_stopping_rounds'])
    mlflow.log_metrics({f'val_{k}': v for k, v in m_val.items()})
    mlflow.log_metrics({f'test_{k}': v for k, v in m_test.items()})
    mlflow.xgboost.log_model(xgb_model, 'xgb_model')

results['XGBoost'] = m_test

plot_predictions(test['Date'], y_test, pred_test_xgb, 'XGBoost — Test 2023–2024')

### 5.5 Importance des features (XGBoost)

In [ ]:
importances = pd.Series(xgb_model.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('Blues_d', len(importances))
importances.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Importance des features — XGBoost', fontweight='bold')
ax.set_xlabel('Importance (gain)')
plt.tight_layout()
plt.show()

### 5.6 Arbre de Décision (Decision Tree)

In [ ]:
from sklearn.tree import DecisionTreeRegressor

DT_PARAMS = {
    'max_depth':        8,
    'min_samples_leaf': 10,
    'random_state':     42,
}

with mlflow.start_run(run_name='DecisionTree'):
    dt = DecisionTreeRegressor(**DT_PARAMS)
    dt.fit(X_train, y_train)

    pred_val_dt  = dt.predict(X_val)
    pred_test_dt = dt.predict(X_test)

    m_val  = evaluate(y_val,  pred_val_dt,  'DecisionTree val')
    m_test = evaluate(y_test, pred_test_dt, 'DecisionTree test')

    mlflow.log_params(DT_PARAMS)
    mlflow.log_metrics({f'val_{k}': v for k, v in m_val.items()})
    mlflow.log_metrics({f'test_{k}': v for k, v in m_test.items()})
    mlflow.sklearn.log_model(dt, 'dt_model')

results['DecisionTree'] = m_test

# Importance des features (arbre)
importances_dt = pd.Series(dt.feature_importances_, index=FEATURES).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(10, 5))
importances_dt.plot(kind='barh', ax=ax, color=sns.color_palette('Greens_d', len(importances_dt)))
ax.set_title('Importance des features — Decision Tree', fontweight='bold')
ax.set_xlabel('Importance (impureté)')
plt.tight_layout()
plt.show()

plot_predictions(test['Date'], y_test, pred_test_dt, 'Decision Tree — Test 2023–2024')

### 5.7 K-Nearest Neighbors (KNN)

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

KNN_PARAMS = {
    'n_neighbors': 10,
    'weights':     'distance',
    'metric':      'euclidean',
    'n_jobs':      -1,
}

with mlflow.start_run(run_name='KNN'):
    pipe_knn = Pipeline([
        ('scaler', StandardScaler()),
        ('model',  KNeighborsRegressor(**KNN_PARAMS))
    ])
    pipe_knn.fit(X_train, y_train)

    pred_val_knn  = pipe_knn.predict(X_val)
    pred_test_knn = pipe_knn.predict(X_test)

    m_val  = evaluate(y_val,  pred_val_knn,  'KNN val')
    m_test = evaluate(y_test, pred_test_knn, 'KNN test')

    mlflow.log_params(KNN_PARAMS)
    mlflow.log_metrics({f'val_{k}': v for k, v in m_val.items()})
    mlflow.log_metrics({f'test_{k}': v for k, v in m_test.items()})
    mlflow.sklearn.log_model(pipe_knn, 'knn_model')

results['KNN'] = m_test

plot_predictions(test['Date'], y_test, pred_test_knn, 'KNN (k=10, distance) — Test 2023–2024')

### 5.8 Réseau de Neurones Artificiels (ANN — MLP)

In [ ]:
import tensorflow as tf
from tensorflow import keras
import time

# Standardisation spécifique au réseau de neurones
scaler_nn = StandardScaler()
X_train_nn = scaler_nn.fit_transform(X_train)
X_val_nn   = scaler_nn.transform(X_val)
X_test_nn  = scaler_nn.transform(X_test)

def build_ann(input_dim):
    model = keras.Sequential([
        keras.layers.Input(shape=(input_dim,)),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dense(1),
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                  loss='mse', metrics=['mae'])
    return model

ANN_PARAMS = {'epochs': 150, 'batch_size': 32, 'patience': 15}

with mlflow.start_run(run_name='ANN'):
    ann = build_ann(X_train_nn.shape[1])

    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=ANN_PARAMS['patience'],
        restore_best_weights=True, verbose=0
    )

    t0 = time.time()
    history = ann.fit(
        X_train_nn, y_train,
        validation_data=(X_val_nn, y_val),
        epochs=ANN_PARAMS['epochs'],
        batch_size=ANN_PARAMS['batch_size'],
        callbacks=[early_stop],
        verbose=0
    )
    training_time = time.time() - t0
    print(f'Entraînement : {len(history.history["loss"])} epochs en {training_time:.1f}s')

    pred_val_ann  = ann.predict(X_val_nn,  verbose=0).flatten()
    pred_test_ann = ann.predict(X_test_nn, verbose=0).flatten()

    m_val  = evaluate(y_val,  pred_val_ann,  'ANN val')
    m_test = evaluate(y_test, pred_test_ann, 'ANN test')

    mlflow.log_params(ANN_PARAMS)
    mlflow.log_metric('training_time_s', round(training_time, 2))
    mlflow.log_metrics({f'val_{k}': v for k, v in m_val.items()})
    mlflow.log_metrics({f'test_{k}': v for k, v in m_test.items()})

results['ANN'] = m_test

# Courbe d'apprentissage
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history.history['loss'],     label='Train loss (MSE)')
ax.plot(history.history['val_loss'], label='Val loss (MSE)')
ax.set_title("ANN — Courbe d'apprentissage", fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE')
ax.legend()
plt.tight_layout()
plt.show()

plot_predictions(test['Date'], y_test, pred_test_ann, 'ANN — Test 2023–2024')

## 6. Comparaison des modèles

In [ ]:
df_results = pd.DataFrame(results).T.sort_values('rmse')
df_results = df_results.rename(columns={'rmse': 'RMSE (MW)', 'mape': 'MAPE (%)', 'r2': 'R²'})
df_results[['RMSE (MW)', 'MAPE (%)', 'R²']] = df_results[['RMSE (MW)', 'MAPE (%)', 'R²']].astype(float)

display(df_results.style
    .format({'RMSE (MW)': '{:.1f}', 'MAPE (%)': '{:.2f}', 'R²': '{:.4f}'})
    .background_gradient(subset=['RMSE (MW)'], cmap='RdYlGn_r')
    .background_gradient(subset=['R²'], cmap='RdYlGn')
)

# Bar chart RMSE
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(df_results.index, df_results['RMSE (MW)'],
               color=sns.color_palette('RdYlGn_r', len(df_results)))
ax.bar_label(bars, fmt='%.0f MW', padding=5)
ax.set_title('RMSE sur le jeu de test (2023–2024)', fontweight='bold')
ax.set_xlabel('RMSE (MW) — moins c\'est mieux')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Analyse des erreurs du meilleur modèle (XGBoost)

In [ ]:
test_eval = test[['Date', 'conso_mean_mw', 'year', 'month', 'dayofweek', 'season']].copy()
test_eval['pred'] = pred_test_xgb
test_eval['error'] = test_eval['conso_mean_mw'] - test_eval['pred']
test_eval['abs_error'] = test_eval['error'].abs()
test_eval['pct_error'] = test_eval['abs_error'] / test_eval['conso_mean_mw'] * 100

print('=== Top 10 pires erreurs ===')
display(test_eval.nlargest(10, 'abs_error')[['Date', 'conso_mean_mw', 'pred', 'error', 'pct_error', 'season']].round(0))

In [ ]:
DOW_NAMES = ['Lun', 'Mar', 'Mer', 'Jeu', 'Ven', 'Sam', 'Dim']
MONTH_NAMES = ['Jan','Fév','Mar','Avr','Mai','Jun','Jul','Aoû','Sep','Oct','Nov','Déc']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MAPE par mois
mape_month = test_eval.groupby('month')['pct_error'].mean()
axes[0].bar(mape_month.index, mape_month.values,
            color=sns.color_palette('coolwarm', 12))
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(MONTH_NAMES)
axes[0].set_title('MAPE par mois (test)', fontweight='bold')
axes[0].set_ylabel('MAPE (%)')

# MAPE par jour de semaine
mape_dow = test_eval.groupby('dayofweek')['pct_error'].mean()
axes[1].bar(mape_dow.index, mape_dow.values,
            color=['steelblue']*5 + ['coral']*2)
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(DOW_NAMES)
axes[1].set_title('MAPE par jour de semaine (test)', fontweight='bold')
axes[1].set_ylabel('MAPE (%)')

plt.tight_layout()
plt.show()

## 8. Sauvegarde du meilleur modèle

In [ ]:
import os, json

os.makedirs('models', exist_ok=True)

# Sauvegarder XGBoost (meilleur modèle basé sur RMSE)
xgb_model.save_model('models/xgb_best.json')

# Sauvegarder la liste des features
with open('models/features.json', 'w') as f:
    json.dump(FEATURES, f, indent=2)

# Identifier le meilleur modèle global (RMSE le plus bas)
best_model_name = min(results, key=lambda k: results[k]['rmse'])
best_metrics    = results[best_model_name]

print(f'Meilleur modèle : {best_model_name}')
print(f'  RMSE = {best_metrics["rmse"]:.1f} MW')
print(f'  MAPE = {best_metrics["mape"]:.2f} %')
print(f'  R²   = {best_metrics["r2"]:.4f}')
print()
print('Modèle XGBoost sauvegardé : models/xgb_best.json')
print('Features sauvegardées      : models/features.json')
print()
print('=== Récapitulatif toutes métriques test ===')
for name, m in sorted(results.items(), key=lambda x: x[1]['rmse']):
    print(f'  {name:<20} RMSE={m["rmse"]:.1f} MW  MAPE={m["mape"]:.2f}%  R²={m["r2"]:.4f}')